# Survey Responses on COVID-19 Vaccination and Menstrual Cycle Changes Among Female University Students Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.mj6q-42w6/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant-structured survey dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.mj6q-42w6/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print the dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We will list the dataset's record sets and, for each, enumerate the fields and associated data columns, by their `@id`. This structure enables precise referencing throughout the notebook.

In [ ]:
# List all record sets, fields, and columns using their `@id`

print("RecordSets in this dataset:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"  - RecordSet @id: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"    name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - Field @id: {field.id}")
        if hasattr(field, 'name'):
            print(f"        name: {field.name}")
        if hasattr(field, 'columns'):
            print("        Columns:")
            for col in field.columns:
                print(f"          - Column @id: {col.id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references use their `@id`.

First, we collect all record set `@id`s for convenience.

In [ ]:
# Extract data from each record set by its @id
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"  No records found for this record set.")

## 4. Exploratory Data Analysis (EDA)
Let's process one record set DataFrame. This section demonstrates filtering, normalization, and grouping based on fields using their `@id`.

_Note: Please adjust the used field `@id` and set the filtering/grouping logic based on your exploration above._

In [ ]:
# For demonstration, select the first record set and first numeric field (if any exists)

if dataframes:
    from pandas.api.types import is_numeric_dtype

    # Pick first non-empty DataFrame
    main_rs_id = next(iter(dataframes))
    df = dataframes[main_rs_id]
    print(f"Sample columns in main record set (@id={main_rs_id}):\n{df.columns.tolist()}")
    # Find first numeric column
    numeric_field = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (z-score) for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field and not is_numeric_dtype(df[col]):
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA. Please check the record set columns above.")
else:
    print("No dataframes loaded from record sets.")

## 5. Visualization
Visualize distribution of a numeric field or summary statistics. Please adjust the field used if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {main_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion

In this notebook, we:
- Loaded and described the FAIR² dataset using the Croissant schema and `mlcroissant`.
- Enumerated all record sets, fields, and columns by their `@id`s.
- Loaded tabular data into DataFrames for direct manipulation.
- Demonstrated basic exploratory analysis including filtering, normalization, and potential grouping by a categorical field.
- Visualized the distribution of a selected numeric field.

Please refer to the dataset schema for authoritative field names (`@id`). For deeper analysis, further data profiling and cleaning is suggested, especially given the dataset's noted missingness and anonymization. For more advanced workflows, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/api/).